# HostWise — Pricing Classification
Compares a host's listed price with the model estimate: **Below Typical**, **Typical**, **Above Typical** or **Uncertain**.
Thresholds come from out-of-fold training predictions only; the regression model is unchanged and the test set is not used. Bands describe where typical listings' prices sit relative to the model estimate, not a fair market value.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "calibration").exists())
GBP_TO_EUR = 1.15
pd.set_option("display.float_format", "{:,.2f}".format)

oof = pd.read_csv(PROJECT_ROOT / "data" / "calibration" / "oof_v22.csv", dtype={"id": "int64"}).set_index("id")
wanted = {"id", "price", "number_of_reviews", "room_type", "estimated_occupancy_l365d", "reviews_per_month"}
data = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "hostwise_5cities.csv",
                   usecols=lambda c: c in wanted, dtype={"id": "int64"}).set_index("id")
cal = oof.join(data, how="left")

train_ids = pd.read_csv(PROJECT_ROOT / "data" / "splits" / "train_ids_eur.csv", dtype={"id": "int64"})["id"]
price = pd.to_numeric(cal["price"].astype(str).str.replace(r"[$,]", "", regex=True), errors="coerce")
assert cal.index.equals(pd.Index(train_ids)), "OOF rows differ from the saved training IDs"
assert np.allclose(price * np.where(cal["city"] == "london", GBP_TO_EUR, 1.0), cal["actual_price_eur"]), "Prices do not match"

cal["ratio"] = cal["actual_price_eur"] / cal["oof_pred_eur"]
cal["has_reviews"] = cal["number_of_reviews"] > 0
cal = cal.drop(columns="price")
print(f"Calibration rows: {len(cal):,} | with reviews: {cal['has_reviews'].mean():.1%} | columns: {list(cal.columns)}")
print(cal["ratio"].describe(percentiles=[.1, .25, .5, .75, .9]).to_string())

Calibration rows: 123,049 | with reviews: 84.4% | columns: ['fold', 'city', 'actual_price_eur', 'oof_pred_eur', 'room_type', 'number_of_reviews', 'estimated_occupancy_l365d', 'reviews_per_month', 'ratio', 'has_reviews']
count   123,049.00
mean          1.12
std           2.94
min           0.04
10%           0.69
25%           0.82
50%           0.98
75%           1.18
90%           1.44
max         742.46


In [2]:
BAND_EDGES = [0, 100, 200, 400, np.inf]
BAND_LABELS = ["<100", "100–200", "200–400", "400+"]
cal["band"] = pd.cut(cal["oof_pred_eur"], BAND_EDGES, labels=BAND_LABELS, right=False)

by_segment = cal.groupby(["band", "has_reviews"], observed=True)["ratio"].agg(
    listings="size", p25=lambda s: s.quantile(.25), median="median", p75=lambda s: s.quantile(.75))
by_segment["width"] = by_segment["p75"] - by_segment["p25"]
display(by_segment)

by_city = cal[cal["has_reviews"]].groupby("city")["ratio"].quantile([.25, .75]).unstack()
by_city["width"] = by_city[.75] - by_city[.25]
display(by_city)

print(f"Estimate percentiles (EUR): 95th {cal['oof_pred_eur'].quantile(.95):,.0f} | 99th {cal['oof_pred_eur'].quantile(.99):,.0f}")

listings  p25  median  p75  width
band    has_reviews                                   
<100    False            2183 0.80    0.98 1.24   0.44
        True            12565 0.84    0.99 1.16   0.32
100–200 False            5671 0.77    0.96 1.19   0.42
        True            41540 0.84    0.98 1.16   0.33
200–400 False            6660 0.75    0.96 1.24   0.49
        True            37060 0.82    0.98 1.18   0.35
400+    False            4624 0.74    0.97 1.28   0.54
        True            12746 0.81    0.98 1.20   0.39

,0.25,0.75,width
city,,,
barcelona,0.85,1.17,0.32
london,0.83,1.18,0.35
madrid,0.82,1.17,0.35
paris,0.82,1.18,0.36
rome,0.84,1.15,0.31


Estimate percentiles (EUR): 95th 632 | 99th 1,201


In [4]:
common = cal[cal["room_type"].isin(["Entire home/apt", "Private room"])]
prices = common.groupby(["city", "room_type", "has_reviews"])["actual_price_eur"].median().unstack()
prices.columns = ["no reviews", "with reviews"]
prices["no / with"] = prices["no reviews"] / prices["with reviews"]
display(prices)

pooled = cal.groupby("has_reviews")["ratio"].quantile([.25, .75]).unstack()
pooled.columns = ["p25", "p75"]
gap = by_segment[["p25", "p75"]] - pooled.reindex(by_segment.index.get_level_values("has_reviews")).to_numpy()
print("Pooled band by reviews:\n", pooled.round(2).to_string())
print(f"Largest gap between any price band and its pooled band: {gap.abs().max().max():.2f}")

no reviews  with reviews  no / with
city      room_type                                           
barcelona Entire home/apt      256.40        276.33       0.93
          Private room         135.25        129.42       1.05
london    Entire home/apt      307.05        274.47       1.12
          Private room          98.90         94.30       1.05
madrid    Entire home/apt      160.85        150.05       1.07
          Private room          80.00         68.50       1.17
paris     Entire home/apt      337.79        233.00       1.45
          Private room         261.16        131.50       1.99
rome      Entire home/apt      209.00        182.00       1.15
          Private room         155.00        114.12       1.36

Pooled band by reviews:
              p25  p75
has_reviews          
False       0.76 1.23
True        0.83 1.17
Largest gap between any price band and its pooled band: 0.05


In [5]:
from datetime import date

ratio_q = cal.groupby("has_reviews")["ratio"].quantile([.01, .25, .75, .99]).unstack()
group_size = cal["has_reviews"].value_counts()


def segment(has_reviews):
    q = ratio_q.loc[has_reviews]
    return {"typical_low": round(q[.25], 3), "typical_high": round(q[.75], 3),
            "extreme_low": round(q[.01], 3), "extreme_high": round(q[.99], 3),
            "listings": int(group_size[has_reviews])}


thresholds = {
    "created": date.today().isoformat(),
    "source": "Out-of-fold predictions of the V2.2 LightGBM/XGBoost pipeline on the training listings (test set not used)",
    "method": "ratio = listed price / model estimate; Typical = 25th-75th percentile of actual/estimate ratios "
              "of training listings in the same review group",
    "note": "Bands describe typical observed prices relative to the model estimate; "
            "they are not fair market values or confidence intervals.",
    "max_reliable_estimate_eur": int(round(cal["oof_pred_eur"].quantile(.99), -2)),
    "segments": {"with_reviews": segment(True), "no_reviews": segment(False)},
}

CONFIG_PATH = PROJECT_ROOT / "models" / "pricing_thresholds.json"
CONFIG_PATH.write_text(json.dumps(thresholds, indent=2))
print(json.dumps(thresholds["segments"], indent=2), "\nmax reliable estimate:", thresholds["max_reliable_estimate_eur"], "EUR")

{
  "with_reviews": {
    "typical_low": 0.829,
    "typical_high": 1.171,
    "extreme_low": 0.474,
    "extreme_high": 2.611,
    "listings": 103911
  },
  "no_reviews": {
    "typical_low": 0.762,
    "typical_high": 1.231,
    "extreme_low": 0.351,
    "extreme_high": 4.917,
    "listings": 19138
  }
} 
max reliable estimate: 1200 EUR


In [6]:
%%writefile {PROJECT_ROOT}/shared_foundation/pricing_classifier.py
"""HostWise pricing classification.

Compares a host's listed nightly price (EUR) with the regression model's estimate.
Thresholds are calibrated in 06_pricing_classification.ipynb from out-of-fold training
predictions and stored in models/pricing_thresholds.json.

Labels describe where the listed price sits relative to typical prices of similar training
listings. They are not fair market values or confidence intervals.
"""
import json
import math
import numbers
from functools import lru_cache
from pathlib import Path

DEFAULT_THRESHOLDS = Path(__file__).resolve().parents[1] / "models" / "pricing_thresholds.json"

BELOW, TYPICAL, ABOVE, UNCERTAIN = "Below Typical", "Typical", "Above Typical", "Uncertain"
GROUP_TEXT = {"with_reviews": "with reviews", "no_reviews": "without reviews"}


@lru_cache(maxsize=None)
def load_thresholds(path=DEFAULT_THRESHOLDS):
    """Read the calibrated thresholds once and cache them."""
    return json.loads(Path(path).read_text())


def _positive_number(value, name):
    if isinstance(value, bool) or not isinstance(value, numbers.Real) or not math.isfinite(value) or value <= 0:
        raise ValueError(f"{name} must be a positive number, got {value!r}")
    return float(value)


def classify_price(predicted_price_eur, listed_price_eur, has_reviews, thresholds=None):
    """Classify a listed nightly price against the model estimate.

    Args:
        predicted_price_eur: model estimate for the listing (EUR per night).
        listed_price_eur: the host's proposed price (EUR per night).
        has_reviews: True if the listing has at least one review.
        thresholds: optional dict; defaults to models/pricing_thresholds.json.

    Returns:
        JSON-serializable dict. Raises ValueError for invalid prices.
    """
    predicted = _positive_number(predicted_price_eur, "predicted_price_eur")
    listed = _positive_number(listed_price_eur, "listed_price_eur")
    config = thresholds or load_thresholds()
    group = "with_reviews" if has_reviews else "no_reviews"
    band = config["segments"][group]

    ratio = listed / predicted
    typical_range = {"lower": round(predicted * band["typical_low"], 2),
                     "upper": round(predicted * band["typical_high"], 2)}
    warnings = []

    if predicted > config["max_reliable_estimate_eur"]:
        status, typical_range = UNCERTAIN, None
        warnings.append(f"The estimate is above {config['max_reliable_estimate_eur']} EUR, "
                        "where the model has too little data for a reliable comparison.")
    elif not band["extreme_low"] <= ratio <= band["extreme_high"]:
        status = UNCERTAIN
        warnings.append("The listed price is far outside the range seen for similar listings; please check the value.")
    elif ratio < band["typical_low"]:
        status = BELOW
    elif ratio > band["typical_high"]:
        status = ABOVE
    else:
        status = TYPICAL

    if not has_reviews:
        warnings.append("No reviews yet: the estimate and range are based on listings without reviews, "
                        "whose prices vary more.")

    return {
        "predicted_price_eur": round(predicted, 2),
        "listed_price_eur": round(listed, 2),
        "typical_range_eur": typical_range,
        "pricing_status": status,
        "difference_eur": round(listed - predicted, 2),
        "difference_percent": round(100 * (listed - predicted) / predicted, 2),
        "has_reviews": bool(has_reviews),
        "basis": f"Typical = middle 50% of training listings {GROUP_TEXT[group]}, relative to their model estimate",
        "warning": " ".join(warnings) or None,
    }

Writing /home/mdalmalki/code/hostwise/shared_foundation/pricing_classifier.py


In [7]:
import sys
sys.path.insert(0, str(PROJECT_ROOT / "shared_foundation"))
from pricing_classifier import classify_price, load_thresholds

load_thresholds.cache_clear()
cal["status"] = [classify_price(p, a, r)["pricing_status"]
                 for p, a, r in zip(cal["oof_pred_eur"], cal["actual_price_eur"], cal["has_reviews"])]

ORDER = ["Below Typical", "Typical", "Above Typical", "Uncertain"]
display(pd.crosstab(cal["has_reviews"], cal["status"], normalize="index").mul(100).reindex(columns=ORDER))

reviewed = cal[cal["has_reviews"]]
behaviour = reviewed.groupby("status")[["estimated_occupancy_l365d", "reviews_per_month"]].median().reindex(ORDER)
behaviour["listings"] = reviewed["status"].value_counts()
behaviour

status,Below Typical,Typical,Above Typical,Uncertain
has_reviews,,,,
False,23.22,48.37,23.00,5.41
True,23.83,49.84,23.83,2.50


,estimated_occupancy_l365d,reviews_per_month,listings
status,,,
Below Typical,70.00,1.03,24762
Typical,83.00,1.19,51788
Above Typical,55.00,0.95,24759
Uncertain,21.00,0.54,2602


In [8]:
import joblib

pipeline = joblib.load(PROJECT_ROOT / "models" / "hostwise_pipeline.joblib")

paris = {
    "city": "paris", "neighbourhood_cleansed": "Hôtel-de-Ville", "room_type": "Entire home/apt",
    "property_type": "Entire rental unit", "accommodates": 4, "bedrooms": 2, "beds": 2,
    "bathrooms_text": "1 bath", "latitude": 48.8566, "longitude": 2.3522, "minimum_nights": 2,
    "review_scores_rating": 4.8, "number_of_reviews": 25, "host_is_superhost": "f",
    "calculated_host_listings_count": 1, "hosts_time_as_host_years": 3,
    "amenities": '["Wifi", "Kitchen", "Washer", "Elevator", "Dedicated workspace", "Heating", "Hair dryer"]',
}
paris_new = {**paris, "number_of_reviews": 0, "review_scores_rating": np.nan}
predict = lambda listing: float(pipeline.predict(pd.DataFrame([listing])[pipeline.feature_names_in_])[0])
est, est_new = predict(paris), predict(paris_new)

cases = {
    "Paris, listed at estimate": (est, est, True),
    "Paris, listed 25% above": (est, est * 1.25, True),
    "Paris, listed 25% below": (est, est * 0.75, True),
    "Paris, no reviews, same listed price": (est_new, est, False),
    "estimate above limit": (1500, 1500, True),
    "listed price typo (x10)": (est, est * 10, True),
}
results = pd.DataFrame([{"case": name, **classify_price(*args)} for name, args in cases.items()]).set_index("case")
display(results[["predicted_price_eur", "listed_price_eur", "typical_range_eur", "pricing_status", "warning"]])

try:
    classify_price(est, -50, True)
except ValueError as error:
    print("Invalid input →", error)

,predicted_price_eur,listed_price_eur,typical_range_eur,pricing_status,warning
case,,,,,
"Paris, listed at estimate",374.07,374.07,"{'lower': 310.1, 'upper': 438.03}",Typical,None
"Paris, listed 25% above",374.07,467.59,"{'lower': 310.1, 'upper': 438.03}",Above Typical,None
"Paris, listed 25% below",374.07,280.55,"{'lower': 310.1, 'upper': 438.03}",Below Typical,None
"Paris, no reviews, same listed price",570.17,374.07,"{'lower': 434.47, 'upper': 701.88}",Below Typical,No reviews yet: the estimate and range are bas...
estimate above limit,"1,500.00","1,500.00",None,Uncertain,"The estimate is above 1200 EUR, where the mode..."
listed price typo (x10),374.07,"3,740.68","{'lower': 310.1, 'upper': 438.03}",Uncertain,The listed price is far outside the range seen...


Invalid input → listed_price_eur must be a positive number, got -50


## Conclusions
- **Method:** ratio = listed price / model estimate. **Typical** = middle 50% of out-of-fold actual/estimate ratios of training listings in the same review group. Test set not used.
- **Segments:** with reviews (Typical 0.83–1.17× estimate) and without reviews (0.76–1.23×). Price level and city change the band by at most ±5%, so they are not used.
- **Uncertain:** estimate above 1,200 EUR (99th percentile of training estimates), or listed price outside the 1st–99th percentile ratio range (likely typo or placeholder price).
- **Review effect:** listings without reviews charge more than reviewed listings in 9 of 10 city/room-type groups (Paris up to ×2). The model reflects this; no artificial review values are used, but these listings get a wider band and a warning.
- **Validation (training OOF):** ~50% Typical per group as designed. *Above Typical* listings have 34% lower median occupancy than *Typical* ones; *Below Typical* listings also book less, so the label means "priced below similar listings", not "could charge more".
- Labels describe typical observed prices relative to the model estimate, **not fair market values**.

## Backend usage
Files: `models/hostwise_pipeline.joblib`, `models/pricing_thresholds.json`, `shared_foundation/pricing_classifier.py` (+ `preprocessing.py`).

```python
import sys, joblib, pandas as pd
sys.path.insert(0, "shared_foundation")
from pricing_classifier import classify_price

pipeline = joblib.load("models/hostwise_pipeline.joblib")
estimate = float(pipeline.predict(pd.DataFrame([listing])[pipeline.feature_names_in_])[0])
result = classify_price(estimate, listed_price_eur, has_reviews=listing["number_of_reviews"] > 0)
```
- `result` is a JSON-ready dict: `predicted_price_eur`, `listed_price_eur`, `typical_range_eur`, `pricing_status`, `difference_eur`, `difference_percent`, `has_reviews`, `basis`, `warning`.
- Invalid prices (missing, zero, negative, not a number) raise `ValueError` → return HTTP 400.
- Always show `warning` when not `null`. London prices are in EUR; convert for display with 1 GBP = 1.15 EUR.
- Thresholds are recalibrated only by re-running this notebook after a model change.